# TREE-BASED METHODS
## A Complete Pipeline Based on "An Introduction to Statistical Learning"

### Analysis Objectives
1. **Decision Trees**: Construction, interpretation and pruning
2. **Bagging**: Variance reduction through bootstrap aggregating
3. **Random Forests**: Bagging improvement with feature sampling
4. **Boosting**: Gradient boosting for bias reduction
5. **Systematic comparison** with previous linear models

### Methodology
- **Consistent dataset**: `aircraft_price_cleaned_log.csv` for comparison with Ridge/Lasso/GAM
- **Cross-validation**: For hyperparameter optimization and pruning
- **Multiple metrics**: RMSE, R², Feature Importance
- **Interpretability**: Partial dependence plots and feature importance
- **Bias-variance analysis**: Comparison between different ensemble methods

In [ ]:
# Complete pipeline for Tree-Based Methods
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, validation_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import partial_dependence, PartialDependenceDisplay
import sklearn
import warnings
warnings.filterwarnings('ignore')

# Tree-based models
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.ensemble import (
    RandomForestRegressor, 
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    AdaBoostRegressor
)

# Plotting configuration
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print("✅ Tree-based methods pipeline imports completed!")
print(f"📈 Pandas version: {pd.__version__}")
print(f"🌳 Scikit-learn version: {sklearn.__version__}")

# Set random seed for reproducibility
np.random.seed(42)

# 1. DATA LOADING & PREPROCESSING

## Using data consistent with previous analyses

In [ ]:
# Data loading - consistent with Ridge/Lasso/GAM
data = pd.read_csv("Data/aircraft_price_cleaned_log.csv")

print("=== DATASET OVERVIEW ===")
print(f"Shape: {data.shape}")
print(f"Target: price (log-transformed)")

print("\n=== FEATURE TYPES ===")
categorical_features = data.select_dtypes(include=['object']).columns.tolist()
numerical_features = data.select_dtypes(include=[np.number]).columns.tolist()
numerical_features.remove('price')  # Remove target

print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"Numerical features ({len(numerical_features)}): {numerical_features}")

print("\n=== TARGET STATISTICS (log-price) ===")
print(data['price'].describe())

# Feature selection for tree models (exclude model_name too granular)
if 'model_name' in categorical_features:
    categorical_features.remove('model_name')
    print(f"\n⚠️  Removed 'model_name' (too granular for tree models)")

print(f"\nFinal features for modeling: {len(categorical_features + numerical_features)}")

# Data preparation
print("\n=== PREPROCESSING FOR TREE MODELS ===")
# Tree models handle categorical variables well with LabelEncoder
X_processed = data[numerical_features].copy()

# Categorical encoding (if present)
if categorical_features:
    for cat_feat in categorical_features:
        le = LabelEncoder()
        X_processed[cat_feat] = le.fit_transform(data[cat_feat])
        print(f"Encoded {cat_feat}: {len(le.classes_)} unique values")

y = data['price'].values
X = X_processed.values
feature_names = X_processed.columns.tolist()

print(f"\nFinal matrix: X{X.shape}, y{y.shape}")
print(f"Feature names: {feature_names}")

In [ ]:
# Train/Test split - consistent with previous analyses
print("=== TRAIN/TEST SPLIT ===")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, shuffle=True
)

print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

# Check target distribution
print(f"\nTarget distribution:")
print(f"Train: mean={y_train.mean():.3f}, std={y_train.std():.3f}")
print(f"Test:  mean={y_test.mean():.3f}, std={y_test.std():.3f}")

# Container for results
tree_results = {}
print("\n✅ Setup completed - ready for tree modeling!")

# 2. DECISION TREES
## Implementation of Algorithm 8.1 (ISLR) - Cost Complexity Pruning

### Theory
- **Cost complexity pruning**: Minimization of $C_\alpha(T) = \sum_{m=1}^{|T|} N_m Q_m(T) + \alpha |T|$
- **Cross-validation**: For selection of complexity parameter $\alpha$
- **Interpretability**: Analysis of decision rules

### Implementation

# 2. DECISION TREES
## Construction, interpretation and pruning following ISLR

In [ ]:
# 2.1 Decision Tree - Complexity analysis (following ISLR Ch 8.1)
print("=== DECISION TREES - COMPLEXITY ANALYSIS ===")

def evaluate_tree_complexity():
    """Analyze the effect of tree depth (ISLR Fig 8.5)"""
    
    max_depths = range(1, 21)
    train_rmse = []
    test_rmse = []
    cv_rmse = []
    
    print("Analyzing tree complexity...")
    
    for depth in max_depths:
        # Tree with fixed depth
        tree = DecisionTreeRegressor(
            max_depth=depth,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=42
        )
        
        # Fit
        tree.fit(X_train, y_train)
        
        # Training error
        y_pred_train = tree.predict(X_train)
        train_rmse.append(np.sqrt(mean_squared_error(y_train, y_pred_train)))
        
        # Test error
        y_pred_test = tree.predict(X_test)
        test_rmse.append(np.sqrt(mean_squared_error(y_test, y_pred_test)))
        
        # Cross-validation error
        cv_scores = cross_val_score(tree, X_train, y_train, cv=5, 
                                   scoring='neg_root_mean_squared_error')
        cv_rmse.append(-cv_scores.mean())
    
    return max_depths, train_rmse, test_rmse, cv_rmse

# Execute complexity analysis
depths, train_errors, test_errors, cv_errors = evaluate_tree_complexity()

# Plot similar to ISLR Figure 8.5.5
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Training vs Test Error
ax1.plot(depths, train_errors, 'o-', label='Training RMSE', color='blue')
ax1.plot(depths, test_errors, 's--', label='Test RMSE', color='red')
ax1.set_xlabel('Tree Depth')
ax1.set_ylabel('RMSE')
ax1.set_title('Training vs Test Error (ISLR Style)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Cross-Validation Error
ax2.plot(depths, cv_errors, 'o-', label='CV RMSE', color='green')
best_depth_idx = np.argmin(cv_errors)
best_depth = depths[best_depth_idx]
ax2.axvline(best_depth, color='red', linestyle='--', 
           label=f'Best depth = {best_depth}')
ax2.set_xlabel('Tree Depth')
ax2.set_ylabel('CV RMSE')
ax2.set_title('Cross-Validation Error')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"🎯 Optimal depth from CV: {best_depth}")
print(f"📊 Test RMSE at optimal depth: {test_errors[best_depth_idx]:.4f}")

# Save the best depth for later comparison
best_depth_from_cv = best_depth
best_rmse_from_depth = test_errors[best_depth_idx]

In [ ]:
# 2.2 Cost Complexity Pruning (ISLR Algorithm 8.1)
print("=== COST COMPLEXITY PRUNING (ISLR Algorithm 8.1) ===")

# Build full tree
full_tree = DecisionTreeRegressor(
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42
)
full_tree.fit(X_train, y_train)

# Cost complexity pruning path
path = full_tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas[:-1]  # Remove last (empty tree)
impurities = path.impurities[:-1]

print(f"📈 Found {len(ccp_alphas)} alpha values for pruning")

# Cross-validation to find best alpha
print("Executing CV for optimal alpha...")

cv_scores_alpha = []
for alpha in ccp_alphas:
    tree_alpha = DecisionTreeRegressor(ccp_alpha=alpha, random_state=42)
    scores = cross_val_score(tree_alpha, X_train, y_train, cv=5,
                            scoring='neg_root_mean_squared_error')
    cv_scores_alpha.append(-scores.mean())

# Find best alpha
best_alpha_idx = np.argmin(cv_scores_alpha)
best_alpha = ccp_alphas[best_alpha_idx]

print(f"🎯 Optimal alpha: {best_alpha:.6f}")

# Train final pruned tree
pruned_tree = DecisionTreeRegressor(ccp_alpha=best_alpha, random_state=42)
pruned_tree.fit(X_train, y_train)

# Evaluation
y_pred_pruned = pruned_tree.predict(X_test)
pruned_rmse = np.sqrt(mean_squared_error(y_test, y_pred_pruned))
pruned_r2 = r2_score(y_test, y_pred_pruned)

print(f"📊 Pruned Tree Test RMSE: {pruned_rmse:.4f}")
print(f"📊 Pruned Tree Test R²: {pruned_r2:.4f}")

# Save results
tree_results['decision_tree'] = {
    'model': 'Decision Tree (Pruned)',
    'rmse': pruned_rmse,
    'r2': pruned_r2,
    'model_object': pruned_tree,
    'best_alpha': best_alpha,
    'tree_depth': pruned_tree.get_depth()
}

# Plot alpha selection (ISLR style)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(ccp_alphas, cv_scores_alpha, 'o-')
plt.axvline(best_alpha, color='red', linestyle='--', 
           label=f'Best α = {best_alpha:.6f}')
plt.xlabel('Alpha (ccp_alpha)')
plt.ylabel('CV RMSE')
plt.title('Cost Complexity Pruning')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
node_counts = [pruned_tree.tree_.node_count for _ in ccp_alphas]
plt.plot(ccp_alphas, impurities, 'o-', label='Impurity')
plt.xlabel('Alpha')
plt.ylabel('Total Impurity')
plt.title('Impurity vs Alpha')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"🌳 Final pruned tree depth: {pruned_tree.get_depth()}")
print(f"🌿 Final pruned tree leaves: {pruned_tree.get_n_leaves()}")

# Comparison between the two approaches
print(f"\n🔍 COMPARISON OF APPROACHES FOR OPTIMIZATION:")
print(f"   📏 Approach 1 - Fixed Depth (GridSearch CV):")
print(f"      • Best depth: {best_depth_from_cv}")
print(f"      • Test RMSE: {best_rmse_from_depth:.4f}")
print(f"   ✂️  Approach 2 - Cost Complexity Pruning:")
print(f"      • Optimal α: {best_alpha:.6f}")
print(f"      • Resulting depth: {pruned_tree.get_depth()}")
print(f"      • Test RMSE: {pruned_rmse:.4f}")

# Determine which approach is better
if pruned_rmse < best_rmse_from_depth:
    print(f"\n🏆 WINNER: Cost Complexity Pruning (RMSE difference: {best_rmse_from_depth - pruned_rmse:.4f})")
else:
    print(f"\n🏆 WINNER: Fixed Depth Approach (RMSE difference: {pruned_rmse - best_rmse_from_depth:.4f})")

# Also train a tree with fixed depth for direct comparison
print(f"\n🔬 Training tree with fixed depth {best_depth_from_cv} for direct comparison...")
fixed_depth_tree = DecisionTreeRegressor(
    max_depth=best_depth_from_cv,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
fixed_depth_tree.fit(X_train, y_train)
y_pred_fixed = fixed_depth_tree.predict(X_test)
fixed_rmse = np.sqrt(mean_squared_error(y_test, y_pred_fixed))
fixed_r2 = r2_score(y_test, y_pred_fixed)

print(f"📊 Fixed Depth Tree (depth={best_depth_from_cv}) - Test RMSE: {fixed_rmse:.4f}, R²: {fixed_r2:.4f}")

# Update results if the fixed depth is better
if fixed_rmse < pruned_rmse:
    print(f"🎯 Using Fixed Depth Tree as final model (better performance)")
    tree_results['decision_tree'] = {
        'model': f'Decision Tree (Fixed Depth={best_depth_from_cv})',
        'rmse': fixed_rmse,
        'r2': fixed_r2,
        'model_object': fixed_depth_tree,
        'best_depth': best_depth_from_cv,
        'tree_depth': fixed_depth_tree.get_depth()
    }
    # Update the variable for subsequent visualizations
    best_tree_for_viz = fixed_depth_tree
    best_tree_type = "Fixed Depth"
else:
    print(f"🎯 Using Pruned Tree as final model (better performance)")
    best_tree_for_viz = pruned_tree
    best_tree_type = "Cost Complexity Pruning"

# Final model evaluation on the test set
print(f"\n🔍 FINAL MODEL EVALUATION ON TEST SET:")
final_model = tree_results['decision_tree']['model_object']
y_pred_final = final_model.predict(X_test)
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred_final))
final_r2 = r2_score(y_test, y_pred_final)

print(f"📊 Final Model (Tree Type: {best_tree_type}) - Test RMSE: {final_rmse:.4f}, R²: {final_r2:.4f}")

# Feature importance extraction
if hasattr(final_model, 'feature_importances_'):
    importances = final_model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)

    print("\n📊 FEATURE IMPORTANCE (from final model):")
    print(feature_importance_df)

    # Plot feature importance
    plt.figure(figsize=(10, 6))
    sns.barplot(data=feature_importance_df, x='Importance', y='Feature')
    plt.title('Feature Importance (Final Model)')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("⚠️ Final model does not have feature importances attribute.")

In [ ]:
# 2.3 Optimal tree visualization (interpretability)
print("=== DECISION TREE VISUALIZATION ===")

# Use the best tree (could be pruned or fixed depth)
best_tree = tree_results['decision_tree']['model_object']
tree_type = tree_results['decision_tree']['model']

print(f"🌳 Visualizing: {tree_type}")

# Plot optimal tree (reasonable size for interpretability)
plt.figure(figsize=(20, 12))
plot_tree(
    best_tree,
    feature_names=feature_names,
    filled=True,
    rounded=True,
    fontsize=10,
    max_depth=4  # Limit ONLY visualization for readability
)

if 'Fixed Depth' in tree_type:
    depth_info = f"Fixed Depth={best_tree.get_depth()}"
else:
    depth_info = f"α={best_alpha:.6f}, depth={best_tree.get_depth()}"
    
plt.title(f'Best Decision Tree ({depth_info})', fontsize=16)
plt.show()

print(f"🌳 OPTIMAL TREE INFORMATION:")
print(f"   • Type: {tree_type}")
print(f"   • Actual depth: {best_tree.get_depth()}")
print(f"   • Number of leaves: {best_tree.get_n_leaves()}")
print(f"   • Total number of nodes: {best_tree.tree_.node_count}")
if 'Pruned' in tree_type:
    print(f"   • Optimal alpha (CV): {best_alpha:.6f}")
else:
    print(f"   • Optimal depth (CV): {best_depth_from_cv}")
print(f"   • Test RMSE: {tree_results['decision_tree']['rmse']:.4f}")
print(f"   • Visualization shows only first 4 levels for readability")

# Decision tree feature importance
feature_importance = pd.Series(
    best_tree.feature_importances_, 
    index=feature_names
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
feature_importance.head(10).plot(kind='bar')
plt.title('Decision Tree - Top 10 Feature Importances')
plt.ylabel('Importance')
plt.xlabel('Features')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("🔍 Top 5 Most Important Features:")
for feat, imp in feature_importance.head().items():
    print(f"  {feat}: {imp:.4f}")

# 3. BAGGING AND RANDOM FORESTS
## Bootstrap Aggregating and Random Forest (ISLR Ch 8.2)

In [ ]:
# 3.1 Random Forest with optimization (ISLR Section 8.2.1)
print("=== RANDOM FOREST OPTIMIZATION ===")

def optimize_random_forest():
    """Optimize Random Forest following ISLR principles"""""
    
    # GridSearch for Random Forest
    rf_param_grid = {
        'n_estimators': [100, 300, 500],
        'max_features': ['sqrt', 'log2', 0.3, 0.5],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }
    
    rf_base = RandomForestRegressor(
        random_state=42,
        n_jobs=-1,
        oob_score=True  # Out-of-bag scoring
    )
    
    print("Optimizing Random Forest...")
    rf_grid = GridSearchCV(
        rf_base,
        rf_param_grid,
        cv=5,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )
    
    rf_grid.fit(X_train, y_train)
    
    return rf_grid

# Random Forest optimization
rf_optimized = optimize_random_forest()
best_rf = rf_optimized.best_estimator_

print(f"🎯 Best RF parameters: {rf_optimized.best_params_}")
print(f"📊 Best CV RMSE: {-rf_optimized.best_score_:.4f}")

# Random Forest evaluation
y_pred_rf = best_rf.predict(X_test)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_r2 = r2_score(y_test, y_pred_rf)
rf_oob_score = best_rf.oob_score_

print(f"🌲 Random Forest Test RMSE: {rf_rmse:.4f}")
print(f"📊 Random Forest Test R²: {rf_r2:.4f}")
print(f"🎯 Random Forest OOB R²: {rf_oob_score:.4f}")

# Save results
tree_results['random_forest'] = {
    'model': 'Random Forest',
    'rmse': rf_rmse,
    'r2': rf_r2,
    'oob_score': rf_oob_score,
    'model_object': best_rf,
    'best_params': rf_optimized.best_params_
}

In [ ]:
# 3.2 Out-of-Bag Error Analysis (ISLR Figure 8.8)
print("=== OUT-OF-BAG ERROR ANALYSIS ===")

def analyze_oob_error():
    """Analyze OOB error vs number of trees (ISLR Figure 8.8)"""
    
    n_estimators_range = range(50, 501, 50)
    oob_errors = []
    test_errors = []
    
    print("Analyzing OOB error vs number of trees...")
    
    for n_est in n_estimators_range:
        # Random Forest with warm_start for efficiency
        rf_oob = RandomForestRegressor(
            n_estimators=n_est,
            max_features=best_rf.max_features,
            max_depth=best_rf.max_depth,
            min_samples_split=best_rf.min_samples_split,
            min_samples_leaf=best_rf.min_samples_leaf,
            random_state=42,
            oob_score=True,
            n_jobs=-1
        )
        
        rf_oob.fit(X_train, y_train)
        
        # OOB Error (1 - OOB Score for error)
        oob_mse = mean_squared_error(y_train, rf_oob.oob_prediction_)
        oob_errors.append(np.sqrt(oob_mse))
        
        # Test Error
        y_pred_test = rf_oob.predict(X_test)
        test_mse = mean_squared_error(y_test, y_pred_test)
        test_errors.append(np.sqrt(test_mse))
    
    return n_estimators_range, oob_errors, test_errors

# Execute OOB analysis
n_trees, oob_rmse, test_rmse = analyze_oob_error()

# Plot OOB vs Test Error (ISLR style)
plt.figure(figsize=(12, 6))
plt.plot(n_trees, oob_rmse, 'o-', label='OOB RMSE', color='blue')
plt.plot(n_trees, test_rmse, 's--', label='Test RMSE', color='red')
plt.xlabel('Number of Trees')
plt.ylabel('RMSE')
plt.title('Random Forest: OOB vs Test Error')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"📈 OOB error stabilizes around {n_trees[np.argmin(oob_rmse[5:]) + 5]} trees")
print(f"🎯 Minimum Test RMSE: {min(test_rmse):.4f} at {n_trees[np.argmin(test_rmse)]} trees")

In [ ]:
# 3.3 max_features parameter comparison (ISLR discussion on m parameter)
print("=== max_features PARAMETER ANALYSIS ===")

def analyze_max_features():
    """Analyze effect of m parameter (max_features) as in ISLR"""
    
    # Different max_features values
    max_features_options = [
        1,  # Extremely Randomized Trees equivalent
        'sqrt',  # Default RF
        'log2',  # Alternative
        0.3,  # 30% of features
        0.5,  # 50% of features
        None  # All features (Bagging)
    ]
    
    results = []
    
    for max_feat in max_features_options:
        rf_feat = RandomForestRegressor(
            n_estimators=300,
            max_features=max_feat,
            max_depth=best_rf.max_depth,
            min_samples_split=best_rf.min_samples_split,
            random_state=42,
            oob_score=True,
            n_jobs=-1
        )
        
        rf_feat.fit(X_train, y_train)
        y_pred = rf_feat.predict(X_test)
        
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        oob_r2 = rf_feat.oob_score_
        
        results.append({
            'max_features': str(max_feat),
            'test_rmse': rmse,
            'oob_r2': oob_r2
        })
        
        feat_count = (max_feat if isinstance(max_feat, int) 
                     else int(np.sqrt(len(feature_names))) if max_feat == 'sqrt'
                     else int(np.log2(len(feature_names))) if max_feat == 'log2'
                     else int(max_feat * len(feature_names)) if isinstance(max_feat, float)
                     else len(feature_names))
        
        print(f"max_features={max_feat} ({feat_count} features): RMSE={rmse:.4f}, OOB_R²={oob_r2:.4f}")
    
    return pd.DataFrame(results)

# max_features analysis
max_feat_results = analyze_max_features()

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Test RMSE
ax1.bar(range(len(max_feat_results)), max_feat_results['test_rmse'])
ax1.set_xticks(range(len(max_feat_results)))
ax1.set_xticklabels(max_feat_results['max_features'], rotation=45)
ax1.set_ylabel('Test RMSE')
ax1.set_title('Test RMSE vs max_features')
ax1.grid(True, alpha=0.3)

# OOB R²
ax2.bar(range(len(max_feat_results)), max_feat_results['oob_r2'])
ax2.set_xticks(range(len(max_feat_results)))
ax2.set_xticklabels(max_feat_results['max_features'], rotation=45)
ax2.set_ylabel('OOB R²')
ax2.set_title('OOB R² vs max_features')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_max_feat = max_feat_results.loc[max_feat_results['test_rmse'].idxmin(), 'max_features']
print(f"🎯 Best max_features: {best_max_feat}")

# 4. BOOSTING
## Gradient Boosting following ISLR Ch 8.2.3

In [ ]:
# 4.1 Gradient Boosting Optimization (ISLR Algorithm 8.2)
print("=== GRADIENT BOOSTING OPTIMIZATION ===")

def optimize_gradient_boosting():
    """Optimize Gradient Boosting following ISLR"""
    
    # GridSearch for Gradient Boosting
    gb_param_grid = {
        'n_estimators': [100, 300, 500, 1000],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'max_depth': [3, 4, 5, 6],
        'subsample': [0.8, 0.9, 1.0],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }
    
    gb_base = GradientBoostingRegressor(
        random_state=42,
        validation_fraction=0.2,  # For early stopping
        n_iter_no_change=10,
        tol=1e-4
    )
    
    print("Optimizing Gradient Boosting...")
    gb_grid = GridSearchCV(
        gb_base,
        gb_param_grid,
        cv=5,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )
    
    gb_grid.fit(X_train, y_train)
    
    return gb_grid

# Gradient Boosting optimization
gb_optimized = optimize_gradient_boosting()
best_gb = gb_optimized.best_estimator_

print(f"🎯 Best GB parameters: {gb_optimized.best_params_}")
print(f"📊 Best CV RMSE: {-gb_optimized.best_score_:.4f}")

# Gradient Boosting evaluation
y_pred_gb = best_gb.predict(X_test)
gb_rmse = np.sqrt(mean_squared_error(y_test, y_pred_gb))
gb_r2 = r2_score(y_test, y_pred_gb)

print(f"🚀 Gradient Boosting Test RMSE: {gb_rmse:.4f}")
print(f"📊 Gradient Boosting Test R²: {gb_r2:.4f}")

# Save results
tree_results['gradient_boosting'] = {
    'model': 'Gradient Boosting',
    'rmse': gb_rmse,
    'r2': gb_r2,
    'model_object': best_gb,
    'best_params': gb_optimized.best_params_
}

In [ ]:
# 4.2 Training vs Validation Error Analysis (ISLR Figure 8.10)
print("=== TRAINING vs VALIDATION ERROR ANALYSIS ===")

# Training history of best model
train_errors = best_gb.train_score_
validation_errors = []

# Calculate validation error for each iteration
for i, y_pred in enumerate(best_gb.staged_predict(X_test)):
    val_mse = mean_squared_error(y_test, y_pred)
    validation_errors.append(val_mse)

# Plot Training vs Validation Error (ISLR style)
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
iterations = range(1, len(train_errors) + 1)
plt.plot(iterations, train_errors, label='Training MSE', color='blue')
plt.plot(iterations, validation_errors[:len(train_errors)], 
         label='Validation MSE', color='red')
plt.xlabel('Boosting Iterations')
plt.ylabel('MSE')
plt.title('Gradient Boosting: Training vs Validation Error')
plt.legend()
plt.grid(True, alpha=0.3)

# Learning rate effect
learning_rates = [0.01, 0.05, 0.1, 0.2, 0.5]
lr_rmse = []

plt.subplot(1, 2, 2)
for lr in learning_rates:
    gb_lr = GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=lr,
        max_depth=best_gb.max_depth,
        random_state=42
    )
    gb_lr.fit(X_train, y_train)
    y_pred_lr = gb_lr.predict(X_test)
    rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
    lr_rmse.append(rmse_lr)

plt.plot(learning_rates, lr_rmse, 'o-')
plt.axvline(best_gb.learning_rate, color='red', linestyle='--',
           label=f'Best LR = {best_gb.learning_rate}')
plt.xlabel('Learning Rate')
plt.ylabel('Test RMSE')
plt.title('Learning Rate Effect')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"🎯 Optimal iterations: {best_gb.n_estimators}")
print(f"📈 Final training MSE: {train_errors[-1]:.4f}")
print(f"📉 Final validation MSE: {validation_errors[len(train_errors)-1]:.4f}")

# 5. FEATURE IMPORTANCE AND INTERPRETABILITY
## Analysis of most important features (ISLR Section 8.2.2)

In [ ]:
# 5.1 Feature Importance Comparison
print("=== FEATURE IMPORTANCE ANALYSIS ===")

# Extract feature importance from all models
def extract_feature_importance():
    """Extract and compare feature importance""""
    
    importance_df = pd.DataFrame(index=feature_names)
    
    # Decision Tree
    dt_model = tree_results['decision_tree']['model_object']
    importance_df['Decision_Tree'] = dt_model.feature_importances_
    
    # Random Forest
    importance_df['Random_Forest'] = best_rf.feature_importances_
    
    # Gradient Boosting
    importance_df['Gradient_Boosting'] = best_gb.feature_importances_
    
    # Bagging (Extra Trees as proxy for pure Bagging)
    bagging_model = ExtraTreesRegressor(
        n_estimators=300,
        max_features=None,  # Use all features (bagging characteristic)
        random_state=42,
        n_jobs=-1
    )
    bagging_model.fit(X_train, y_train)
    importance_df['Bagging'] = bagging_model.feature_importances_
    
    return importance_df, bagging_model

importance_comparison, bagging_model = extract_feature_importance()

# Save bagging model for later comparison
y_pred_bagging = bagging_model.predict(X_test)
bagging_rmse = np.sqrt(mean_squared_error(y_test, y_pred_bagging))
bagging_r2 = r2_score(y_test, y_pred_bagging)

tree_results['bagging'] = {
    'model': 'Bagging (Extra Trees)',
    'rmse': bagging_rmse,
    'r2': bagging_r2,
    'model_object': bagging_model
}

print(f"🌲 Bagging Test RMSE: {bagging_rmse:.4f}")
print(f"📊 Bagging Test R²: {bagging_r2:.4f}")

# Feature importance visualization - 4 separate plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Decision Tree
ax1 = axes[0, 0]
dt_importance = importance_comparison['Decision_Tree'].nlargest(12)
dt_importance.plot(kind='bar', ax=ax1, color='skyblue')
ax1.set_title('Decision Tree - Feature Importance')
ax1.set_ylabel('Importance')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3)

# Plot 2: Random Forest
ax2 = axes[0, 1]
rf_importance = importance_comparison['Random_Forest'].nlargest(12)
rf_importance.plot(kind='bar', ax=ax2, color='green')
ax2.set_title('Random Forest - Feature Importance')
ax2.set_ylabel('Importance')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True, alpha=0.3)

# Plot 3: Bagging
ax3 = axes[1, 0]
bagging_importance = importance_comparison['Bagging'].nlargest(12)
bagging_importance.plot(kind='bar', ax=ax3, color='orange')
ax3.set_title('Bagging - Feature Importance')
ax3.set_ylabel('Importance')
ax3.tick_params(axis='x', rotation=45)
ax3.grid(True, alpha=0.3)

# Plot 4: Gradient Boosting
ax4 = axes[1, 1]
gb_importance = importance_comparison['Gradient_Boosting'].nlargest(12)
gb_importance.plot(kind='bar', ax=ax4, color='red')
ax4.set_title('Gradient Boosting - Feature Importance')
ax4.set_ylabel('Importance')
ax4.tick_params(axis='x', rotation=45)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print top features
print("\n🔍 TOP 5 FEATURES PER MODEL:")
for model in importance_comparison.columns:
    print(f"\n{model}:")
    top_5 = importance_comparison[model].nlargest(5)
    for feat, imp in top_5.items():
        print(f"  {feat}: {imp:.4f}")

# General comparison of most important featureses
print("\n📊 FEATURE IMPORTANCE SUMMARY:")
avg_importance = importance_comparison.mean(axis=1).sort_values(ascending=False)
print("\nTop 10 features (average across all models):")
for feat, imp in avg_importance.head(10).items():
    print(f"  {feat}: {imp:.4f}")

In [ ]:
# 5.2 Partial Dependence Plots (ISLR Figure 8.12)
print("=== PARTIAL DEPENDENCE PLOTS ===")

# Select top 4 features from Random Forest (best model)
top_4_features = importance_comparison['Random_Forest'].nlargest(4).index.tolist()
top_4_indices = [feature_names.index(feat) for feat in top_4_features]

print(f"📊 Creating partial dependence plots for: {top_4_features}")

# Partial dependence plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, (feat_name, feat_idx) in enumerate(zip(top_4_features, top_4_indices)):
    # Use Random Forest for partial dependence
    PartialDependenceDisplay.from_estimator(
        best_rf, 
        X_train, 
        [feat_idx],
        feature_names=feature_names,
        ax=axes[i],
        kind='average'
    )
    axes[i].set_title(f'Partial Dependence: {feat_name}')

plt.tight_layout()
plt.show()

# Interaction effects (if enough features available)
if len(top_4_features) >= 2:
    print("\n📈 INTERACTION EFFECTS (Top 2 features):")
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    PartialDependenceDisplay.from_estimator(
        best_rf,
        X_train,
        [top_4_indices[:2]],  # First 2 features
        feature_names=feature_names,
        ax=ax,
        kind='average'
    )
    plt.title(f'Interaction: {top_4_features[0]} vs {top_4_features[1]}', fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
# 5.4 Complete Performance Comparison of all Tree Models
print("=== COMPLETE TREE MODELS COMPARISON ===")

# Function to calculate delogged RMSE
def calculate_delogged_rmse(y_true_log, y_pred_log):
    """Calculate RMSE in original scale (dollars) from log-transformed predictions"""
    y_true_original = np.exp(y_true_log)
    y_pred_original = np.exp(y_pred_log)
    return np.sqrt(mean_squared_error(y_true_original, y_pred_original))

# Extract metrics from all tree-based models
models_comparison = []

for model_key, results in tree_results.items():
    model_name = results['model']
    rmse = results['rmse']
    r2 = results['r2']
    
    # Calculate delogged RMSE
    model_obj = results['model_object']
    y_pred_log = model_obj.predict(X_test)
    rmse_delogged = calculate_delogged_rmse(y_test, y_pred_log)
    
    # Add additional information for each model
    extra_info = ""
    if model_key == 'decision_tree':
        if 'Fixed Depth' in model_name:
            extra_info = f" (depth={results.get('best_depth', 'N/A')})"
        else:
            extra_info = f" (α={results.get('best_alpha', 'N/A'):.4f})"
    elif model_key == 'random_forest':
        extra_info = f" (trees={results['best_params'].get('n_estimators', 'N/A')})"
    elif model_key == 'gradient_boosting':
        extra_info = f" (lr={results['best_params'].get('learning_rate', 'N/A')})"
    
    models_comparison.append({
        'Model': model_name + extra_info,
        'RMSE': rmse,
        'RMSE_Delogged': rmse_delogged,
        'R²': r2,
        'Model_Type': model_key.replace('_', ' ').title()
    })

# Create DataFrame and sort by RMSE
comparison_df = pd.DataFrame(models_comparison)
comparison_df = comparison_df.sort_values('RMSE')

print("📊 PERFORMANCE COMPARISON (sorted by RMSE):")
print("=" * 85)
for idx, row in comparison_df.iterrows():
    print(f"{row['Model']:<40} | RMSE: {row['RMSE']:.4f} | RMSE($): ${row['RMSE_Delogged']:,.0f} | R²: {row['R²']:.4f}")

print("\n🏆 RANKING:")
for i, (idx, row) in enumerate(comparison_df.iterrows(), 1):
    emoji = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else f"{i}️⃣"
    print(f"   {emoji} {row['Model']}: RMSE = {row['RMSE']:.4f} (${row['RMSE_Delogged']:,.0f})")

# Comparative visualization with 2 subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Colors for each model type
colors = {
    'Decision Tree': 'lightblue',
    'Random Forest': 'lightgreen', 
    'Gradient Boosting': 'orange',
    'Bagging': 'pink'
}
bar_colors = [colors.get(row['Model_Type'], 'gray') for _, row in comparison_df.iterrows()]

# Plot 1: RMSE Delogged (dollars)
bars1 = ax1.bar(range(len(comparison_df)), comparison_df['RMSE_Delogged'], color=bar_colors)
ax1.set_xticks(range(len(comparison_df)))
ax1.set_xticklabels([name.split('(')[0].strip() for name in comparison_df['Model']], 
                    rotation=45, ha='right', fontsize=12)
ax1.set_ylabel('RMSE (Dollars)', fontsize=14)
ax1.set_title('Tree Models: RMSE (Original Scale)', fontsize=16)
ax1.grid(axis='y', alpha=0.3)

# Add values above the bars (dollar format)
for bar, rmse_delogged in zip(bars1, comparison_df['RMSE_Delogged']):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, height + max(comparison_df['RMSE_Delogged'])*0.01,
             f'${rmse_delogged:,.0f}', ha='center', va='bottom', fontsize=10, rotation=45)

# Plot 2: R² Comparison  
bars2 = ax2.bar(range(len(comparison_df)), comparison_df['R²'], color=bar_colors)
ax2.set_xticks(range(len(comparison_df)))
ax2.set_xticklabels([name.split('(')[0].strip() for name in comparison_df['Model']], 
                    rotation=45, ha='right', fontsize=12)
ax2.set_ylabel('R²', fontsize=14)
ax2.set_title('Tree Models: R²', fontsize=16)
ax2.grid(axis='y', alpha=0.3)

# Add values above the bars
for bar, r2 in zip(bars2, comparison_df['R²']):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, height + 0.005,
             f'{r2:.4f}', ha='center', va='bottom', fontsize=10)

# Legend
legend_elements = [plt.Rectangle((0,0),1,1, facecolor=color, label=model_type) 
                  for model_type, color in colors.items() if model_type in comparison_df['Model_Type'].values]
ax2.legend(handles=legend_elements, loc='lower right', fontsize=12)

plt.tight_layout()
plt.show()

# Analysis of differences
best_model = comparison_df.iloc[0]
worst_model = comparison_df.iloc[-1]
rmse_improvement = worst_model['RMSE'] - best_model['RMSE']
rmse_improvement_pct = (rmse_improvement / worst_model['RMSE']) * 100

# Delogged RMSE analysis
rmse_delogged_improvement = worst_model['RMSE_Delogged'] - best_model['RMSE_Delogged']
rmse_delogged_improvement_pct = (rmse_delogged_improvement / worst_model['RMSE_Delogged']) * 100

print(f"\n📈 PERFORMANCE ANALYSIS:")
print(f"   🏆 Best model: {best_model['Model']}")
print(f"   📊 RMSE range (log): {best_model['RMSE']:.4f} - {worst_model['RMSE']:.4f}")
print(f"   💰 RMSE range ($): ${best_model['RMSE_Delogged']:,.0f} - ${worst_model['RMSE_Delogged']:,.0f}")
print(f"   🚀 Improvement (log): {rmse_improvement:.4f} ({rmse_improvement_pct:.1f}%)")
print(f"   💵 Improvement ($): ${rmse_delogged_improvement:,.0f} ({rmse_delogged_improvement_pct:.1f}%)")

# Comparison with target variance
target_std = np.std(y_test)
target_std_delogged = np.std(np.exp(y_test))
print(f"   📏 Target std (log): {target_std:.4f}")
print(f"   💲 Target std ($): ${target_std_delogged:,.0f}")
print(f"   🎯 Best RMSE vs Target std: {(best_model['RMSE']/target_std)*100:.1f}%")
print(f"   💎 Best RMSE($) vs Target std($): {(best_model['RMSE_Delogged']/target_std_delogged)*100:.1f}%")

# Practical interpretation
y_test_mean_dollars = np.exp(y_test).mean()
print(f"\n🔍 PRACTICAL INTERPRETATION:")
print(f"   • The best model has an average error of ~${best_model['RMSE_Delogged']:,.0f} in price predictions")
print(f"   • This represents {(best_model['RMSE_Delogged']/y_test_mean_dollars)*100:.1f}% of the average aircraft price")
print(f"   • The approximate confidence interval is ±${best_model['RMSE_Delogged']*1.96:,.0f} (95%)")

# Save statistics for future comparisons
delogged_stats = {
    'best_model_name': best_model['Model'],
    'best_rmse_dollars': best_model['RMSE_Delogged'],
    'target_mean_dollars': y_test_mean_dollars,
    'target_std_dollars': target_std_delogged
}

print(f"\n✅ Delogged RMSE analysis completed!")


In [ ]:
import matplotlib.pyplot as plt

# Prepara etichette e colori
model_labels = [name.split('(')[0].strip() for name in comparison_df['Model']]
bar_colors  = [colors.get(t, 'gray') for t in comparison_df['Model_Type']]

# === FIGURA 1: RMSE (Original Scale) ===
fig_rmse, ax_rmse = plt.subplots(figsize=(10, 6))
bars_rmse = ax_rmse.bar(range(len(comparison_df)), comparison_df['RMSE_Delogged'], color=bar_colors)

ax_rmse.set_xticks(range(len(comparison_df)))
ax_rmse.set_xticklabels(model_labels, rotation=45, ha='right', fontsize=12)
ax_rmse.set_ylabel('RMSE (Dollars)', fontsize=14)
ax_rmse.set_title('Tree Models: RMSE (Original Scale)', fontsize=16)
ax_rmse.grid(axis='y', alpha=0.3)

# Valori sopra le barre
for bar, val in zip(bars_rmse, comparison_df['RMSE_Delogged']):
    ax_rmse.text(
        bar.get_x() + bar.get_width()/2,
        val + max(comparison_df['RMSE_Delogged'])*0.01,
        f'${val:,.0f}',
        ha='center', va='bottom', fontsize=10, rotation=45
    )

plt.tight_layout()
plt.show()


# === FIGURA 2: R² Comparison ===
fig_r2, ax_r2 = plt.subplots(figsize=(10, 6))
bars_r2 = ax_r2.bar(range(len(comparison_df)), comparison_df['R²'], color=bar_colors)

ax_r2.set_xticks(range(len(comparison_df)))
ax_r2.set_xticklabels(model_labels, rotation=45, ha='right', fontsize=12)
ax_r2.set_ylabel('R²', fontsize=14)
ax_r2.set_title('Tree Models: R² Comparison', fontsize=16)
ax_r2.grid(axis='y', alpha=0.3)

# Valori sopra le barre
for bar, val in zip(bars_r2, comparison_df['R²']):
    ax_r2.text(
        bar.get_x() + bar.get_width()/2,
        val + 0.005,
        f'{val:.4f}',
        ha='center', va='bottom', fontsize=10
    )

# Aggiungi legenda (facoltativa)
legend_handles = [
    plt.Rectangle((0,0),1,1, facecolor=col, label=mt)
    for mt, col in colors.items() if mt in comparison_df['Model_Type'].values
]
ax_r2.legend(handles=legend_handles, loc='lower right', fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
# 6.2 Complete Residual Analysis for All Models
print("=== COMPLETE RESIDUAL ANALYSIS ===\n")

import scipy.stats
from scipy.stats import jarque_bera, shapiro, normaltest

# Function to calculate residual statistics
def calculate_residual_stats(y_true, y_pred, model_name):
    """Calculate complete statistics on residuals"""
    residuals = y_true - y_pred
    
    stats_dict = {
        'model': model_name,
        'residuals': residuals,
        'mean': np.mean(residuals),
        'std': np.std(residuals),
        'skewness': scipy.stats.skew(residuals),
        'kurtosis': scipy.stats.kurtosis(residuals),
        'min': np.min(residuals),
        'max': np.max(residuals),
        'range': np.max(residuals) - np.min(residuals)
    }
    
    # Normality tests
    _, jb_p = jarque_bera(residuals)
    _, shapiro_p = shapiro(residuals[:5000] if len(residuals) > 5000 else residuals)  # Shapiro max 5000 samples
    _, normaltest_p = normaltest(residuals)
    
    stats_dict.update({
        'jarque_bera_p': jb_p,
        'shapiro_p': shapiro_p,
        'normaltest_p': normaltest_p
    })
    
    return stats_dict

# Calculate residuals for all models
residual_analysis = []
model_predictions = {}

for model_key, results in tree_results.items():
    model_obj = results['model_object']
    model_name = results['model']
    
    # Predictions
    y_pred = model_obj.predict(X_test)
    model_predictions[model_key] = y_pred
    
    # Calculate residual statistics
    residual_stats = calculate_residual_stats(y_test, y_pred, model_name)
    residual_analysis.append(residual_stats)

print("📋 RESIDUAL STATISTICS:")
print("=" * 90)
print(f"{'Model':<25} {'Mean':<8} {'Std':<8} {'Skew':<8} {'Kurt':<8} {'Range':<10} {'JB_p':<8} {'SW_p':<8}")
print("=" * 90)

for stats in residual_analysis:
    print(f"{stats['model']:<25} {stats['mean']:<8.4f} {stats['std']:<8.4f} "
          f"{stats['skewness']:<8.3f} {stats['kurtosis']:<8.3f} {stats['range']:<10.3f} "
          f"{stats['jarque_bera_p']:<8.4f} {stats['shapiro_p']:<8.4f}")

print("\n🗒 INTERPRETATION:")
print("  • Mean ≈ 0: unbiased residuals")
print("  • Skew ≈ 0: symmetric distribution")
print("  • Kurt ≈ 0: normal tails (>0 = heavy tails)")
print("  • JB_p/SW_p > 0.05: normal residuals")

# Identify the model with the most 'normal' residuals
normality_scores = []
for stats in residual_analysis:
    # Score based on: closeness to normality (combination of tests + skewness + kurtosis)
    normal_score = (stats['jarque_bera_p'] + stats['shapiro_p']) / 2 - abs(stats['skewness']) - abs(stats['kurtosis'])
    normality_scores.append((stats['model'], normal_score))

best_residuals = max(normality_scores, key=lambda x: x[1])
print(f"\n🏆 BEST RESIDUALS (normality): {best_residuals[0]} (score: {best_residuals[1]:.3f})")

# Setup colors for consistency
model_colors = {
    'decision_tree': '#FF6B6B',    # Red
    'random_forest': '#4ECDC4',    # Teal
    'gradient_boosting': '#45B7D1', # Blue
    'bagging': '#96CEB4'           # Green
}

# 1. RESIDUALS vs FITTED VALUES (4 subplots)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, (model_key, stats) in enumerate(zip(tree_results.keys(), residual_analysis)):
    y_pred = model_predictions[model_key]
    residuals = stats['residuals']
    model_name = stats['model']
    color = model_colors[model_key]
    
    # Scatter plot residuals vs fitted
    axes[i].scatter(y_pred, residuals, alpha=0.6, color=color, s=30)
    axes[i].axhline(y=0, color='red', linestyle='--', linewidth=2)
    
    axes[i].set_xlabel('Fitted Values (log-price)', fontsize=16)
    axes[i].set_ylabel('Residuals', fontsize=16)
    axes[i].set_title(f'{model_name}\nRMSE: {np.sqrt(np.mean(residuals**2)):.4f}', fontsize=16)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Residuals vs Fitted Values - All Models', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# 2. Q-Q PLOTS for normality (4 subplots)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, (model_key, stats) in enumerate(zip(tree_results.keys(), residual_analysis)):
    residuals = stats['residuals']
    model_name = stats['model']
    color = model_colors[model_key]
    
    # Q-Q plot
    scipy.stats.probplot(residuals, dist="norm", plot=axes[i])
    axes[i].get_lines()[0].set_markerfacecolor(color)
    axes[i].get_lines()[0].set_markeredgecolor(color)
    axes[i].get_lines()[0].set_markersize(4)
    axes[i].get_lines()[1].set_color('red')
    axes[i].get_lines()[1].set_linewidth(2)
    
    axes[i].set_title(f'{model_name}\nShapiro p-value: {stats["shapiro_p"]:.4f}', fontsize=16)
    axes[i].grid(True, alpha=0.3)
    

plt.suptitle('Q-Q Plots - Normality Test of Residuals', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# 3. RESIDUAL DISTRIBUTION (4 subplots)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, (model_key, stats) in enumerate(zip(tree_results.keys(), residual_analysis)):
    residuals = stats['residuals']
    model_name = stats['model']
    color = model_colors[model_key]
    
    # Histogram + normal curve
    axes[i].hist(residuals, bins=30, density=True, alpha=0.7, color=color, edgecolor='black')
    
    # Overlay theoretical normal distribution
    x_norm = np.linspace(residuals.min(), residuals.max(), 100)
    y_norm = scipy.stats.norm.pdf(x_norm, stats['mean'], stats['std'])
    axes[i].plot(x_norm, y_norm, 'r-', linewidth=2, label='Normal')
    
    axes[i].set_xlabel('Residuals', fontsize=12)
    axes[i].set_ylabel('Density', fontsize=12)
    
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Residual Distribution vs Theoretical Normal', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ Residual visualization completed!")